# 人体动作识别 — 多分支 1D-CNN

使用 UCI HAR Dataset，通过三个不同卷积核大小（3/5/7）的多头一维卷积神经网络识别 6 种日常动作。多分支设计可同时捕获不同时间尺度的运动模式。

## 1. 导入依赖

- `numpy` / `pandas`：数据处理
- `tensorflow.keras`：构建 1D-CNN 模型
- `sklearn`：标准化

In [1]:
import glob, os
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from tensorflow.keras import Input, Model
from tensorflow.keras.layers import (
    Conv1D, Dense, Dropout, MaxPooling1D,
    BatchNormalization, GlobalAveragePooling1D, concatenate
)
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2


## 2. 超参数配置

- `N_CLASSES = 6`：6 种日常动作
- `BATCH_SIZE = 64`：每批 64 条样本
- `MAX_EPOCHS = 100`：最多训练 100 轮（早停会提前结束）

In [2]:
N_CLASSES   = 6

BATCH_SIZE  = 64
MAX_EPOCHS  = 100

## 3. 数据加载

UCI HAR 数据集中，每组（train/test）目录结构如下：
- `Inertial Signals/` 下 9 个 `.txt` 文件，每个对应一种传感器通道
- `y_train.txt` / `y_test.txt` 为标签（1~6）

`np.stack` 将 9 个 `(样本数, 128)` 的通道文件沿最后一维堆叠为 `(样本数, 128, 9)`。

In [3]:
def load_dataset(data_dir):
    # 获取 Inertial Signals 目录下所有 txt 文件，排序保证通道顺序一致
    files = sorted(glob.glob(os.path.join(data_dir, 'Inertial Signals', '*.txt')))
    # 逐个读入 9 个传感器文件，沿最后一维堆叠为 (N, 128, 9)
    X = np.stack([np.loadtxt(f, dtype=np.float32) for f in files], axis=-1)
    # 从路径末尾提取 'train' 或 'test'
    group = os.path.basename(data_dir)
    # 读取标签文件 y_train.txt 或 y_test.txt
    y = np.loadtxt(os.path.join(data_dir, f'y_{group}.txt'), dtype=np.float32)
    print(f'{group}: X {X.shape}, y {y.shape}')
    return X, y

## 4. 模型结构 — 多分支 1D-CNN

三个并行的卷积分支，卷积核大小分别为 **3、5、7**，从不同时间窗口提取特征：

```
输入 (128, 9)
  ├─ Conv1D(64, 3) → Conv1D(64, 3) → MaxPool → Conv1D(128, 3) → MaxPool → GAP
  ├─ Conv1D(64, 5) → Conv1D(64, 5) → MaxPool → Conv1D(128, 5) → MaxPool → GAP
  └─ Conv1D(64, 7) → Conv1D(64, 7) → MaxPool → Conv1D(128, 7) → MaxPool → GAP
       ↓
  Concat → Dense(128) → Dropout(0.5) → Dense(64) → Dropout(0.5) → Softmax(6)
```

每层卷积后接 `BatchNormalization` 加速收敛，`l2(1e-4)` 正则化防过拟合。

In [4]:
def build_model(input_shape):
    inputs = Input(shape=input_shape)
    branches = []
    for k in [3, 5, 7]:
        x = Conv1D(64, k, activation='relu', padding='same',
                   kernel_regularizer=l2(1e-3))(inputs)
        x = BatchNormalization()(x)
        x = Conv1D(64, k, activation='relu', padding='same',
                   kernel_regularizer=l2(1e-3))(x)
        x = BatchNormalization()(x)
        x = MaxPooling1D(2)(x)
        x = Dropout(0.3)(x)
        x = Conv1D(128, k, activation='relu', padding='same',
                   kernel_regularizer=l2(1e-3))(x)
        x = BatchNormalization()(x)
        x = MaxPooling1D(2)(x)
        x = Dropout(0.3)(x)
        x = GlobalAveragePooling1D()(x)
        branches.append(x)

    x = concatenate(branches)
    x = Dense(128, activation='relu', kernel_regularizer=l2(1e-3))(x)
    x = BatchNormalization()(x)
    x = Dropout(0.5)(x)
    x = Dense(64, activation='relu', kernel_regularizer=l2(1e-3))(x)
    x = BatchNormalization()(x)
    x = Dropout(0.5)(x)
    outputs = Dense(N_CLASSES, activation='softmax')(x)

    model = Model(inputs, outputs)
    model.compile(loss='categorical_crossentropy',
                  optimizer=tf.keras.optimizers.Adam(1e-3),
                  metrics=['accuracy'])
    return model

## 5. 训练与评估

用全部训练集训练，留出 20% 做验证，早停后评估测试集。

In [ ]:
X_train, y_train = load_dataset('./UCI HAR Dataset/train')
X_test,  y_test  = load_dataset('./UCI HAR Dataset/test')

# 标准化
scaler = StandardScaler().fit(X_train.reshape(-1, X_train.shape[-1]))
X_train = scaler.transform(X_train.reshape(-1, X_train.shape[-1])).reshape(X_train.shape)
X_test  = scaler.transform(X_test.reshape(-1, X_test.shape[-1])).reshape(X_test.shape)

# one-hot 编码
y_train_ohe = to_categorical(y_train - 1)
y_test_ohe  = to_categorical(y_test - 1)

model = build_model(X_train.shape[1:])
model.fit(
    X_train, y_train_ohe,
    epochs=MAX_EPOCHS, batch_size=BATCH_SIZE, validation_split=0.2,
    callbacks=[
        EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6),
    ],
    verbose=1,
)

_, test_acc = model.evaluate(X_test, y_test_ohe, verbose=0)
print(f'测试集准确率: {test_acc * 100:.2f}%')

train: X (7352, 128, 9), y (7352,)
test: X (2947, 128, 9), y (2947,)
Epoch 1/100
92/92 [==============================] - 5s 14ms/step - loss: 1.2874 - accuracy: 0.8271 - val_loss: 1.4167 - val_accuracy: 0.8219 - lr: 0.0010
Epoch 2/100
92/92 [==============================] - 1s 11ms/step - loss: 0.9338 - accuracy: 0.9308 - val_loss: 1.1580 - val_accuracy: 0.8355 - lr: 0.0010
Epoch 3/100
92/92 [==============================] - 1s 14ms/step - loss: 0.8307 - accuracy: 0.9481 - val_loss: 0.9600 - val_accuracy: 0.9109 - lr: 0.0010
Epoch 4/100
92/92 [==============================] - 1s 13ms/step - loss: 0.7633 - accuracy: 0.9485 - val_loss: 0.8901 - val_accuracy: 0.9089 - lr: 0.0010
Epoch 5/100
92/92 [==============================] - 1s 13ms/step - loss: 0.7001 - accuracy: 0.9475 - val_loss: 0.8005 - val_accuracy: 0.9341 - lr: 0.0010
Epoch 6/100
92/92 [==============================] - 1s 13ms/step - loss: 0.6252 - accuracy: 0.9543 - val_loss: 0.7472 - val_accuracy: 0.9259 - lr: 0.0010
E

: 